In [0]:
!pip install torch 

In [0]:
# using held out test set, conduct sub group evaluation on: race, ethnicity, care_site, outcome
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

CFG = {
    "export_path": "/path/to/exports/full_analysis_export.pkl",
    "cohort_table": "catalog.schema.step7_filtered_cohort_with_apos",
    "person_table": "catalog.omop.person",
    "features": ["race", "ethnicity", "care_site_id"],
    "threshold": 0.5,                  # same for all classes; used by f1 and balanced accuracy
    "metrics": ["auroc", "auprc", "f1", "balanced_acc", "brier"],
    "n_boot": 1000,
    "seed": 42,
    "ci": 0.95,
    "min_group_n": 10,                 # skip groups smaller than this
    "min_pos": 5,                      # skip a (group, class) with fewer positives
    "min_valid_frac": 0.9,             # fraction of non-degenerate resamples needed to report a CI/p
    "alpha": 0.05,
    "correction": "holm",              # "holm" (controls FWER) or "bh" (controls FDR)
}

with open(CFG["export_path"], "rb") as f:
    exp = pickle.load(f)


In [0]:
from sklearn.metrics import roc_auc_score, average_precision_score

names = list(exp["CLASS_NAMES"])
pids = np.asarray(exp["pids_test"])
p = np.asarray(exp["p_test"])
y = np.asarray(exp["Y_fusion_test"])
assert p.shape == y.shape == (len(pids), len(names)), (p.shape, y.shape, len(pids), len(names))
assert len(set(pids.tolist())) == len(pids), "pids_test has duplicates, so it is not patient-level"

# --- attributes from Spark; assumes person_id == int(pid) (change the cast here if not) ---
pid_int = [int(x) for x in pids]

cohort_sdf = spark.table(CFG["cohort_table"])
cohort = (cohort_sdf.filter(cohort_sdf.person_id.isin(pid_int))
          .select("person_id", "race", "ethnicity").toPandas())
person_sdf = spark.table(CFG["person_table"])
person = (person_sdf.filter(person_sdf.person_id.isin(pid_int))
          .select("person_id", "care_site_id").toPandas())

# a duplicate person_id would silently duplicate patients in the join and
# inflate n, so warn and keep the first row
for nm, d in [("cohort", cohort), ("person", person)]:
    if d["person_id"].duplicated().any():
        print(f"WARNING: duplicate person_id in {nm} table; keeping first row per person")
cohort = cohort.drop_duplicates("person_id")
person = person.drop_duplicates("person_id")

# left-merge onto the test-set order so row j still corresponds to pids[j]
attrs = (pd.DataFrame({"person_id": pid_int})
         .merge(cohort, on="person_id", how="left")
         .merge(person, on="person_id", how="left"))
assert len(attrs) == len(pids)

# missing values become their own explicit group instead of silently dropping patients
for col in CFG["features"]:
    attrs[col] = attrs[col].astype("string").fillna("Missing")

# one patient-level table: attributes + predictions + labels
df = attrs.copy()
for i, n in enumerate(names):
    df[f"pred_{n}"] = p[:, i]
    df[f"y_{n}"] = y[:, i]

print(df.shape)
for col in CFG["features"]:
    print(f"\n{col}:\n{df[col].value_counts().to_string()}")

In [0]:
# def metric_suite(yt, pr, thr):
#     """Metrics for one (group, class). NaN where undefined."""
#     out = {m: np.nan for m in CFG["metrics"]}
#     npos = yt.sum(); nneg = len(yt) - npos
#     if npos > 0 and nneg > 0:                       # AUROC/AUPRC need both labels present
#         out["auroc"] = roc_auc_score(yt, pr)
#         out["auprc"] = average_precision_score(yt, pr)
#     yh = pr >= thr
#     tp = int((yh & (yt == 1)).sum()); fp = int((yh & (yt == 0)).sum())
#     fn = int((~yh & (yt == 1)).sum()); tn = int((~yh & (yt == 0)).sum())
#     sens = tp / (tp + fn) if (tp + fn) else np.nan
#     spec = tn / (tn + fp) if (tn + fp) else np.nan
#     out["f1"] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
#     out["balanced_acc"] = (sens + spec) / 2         # NaN if either is undefined
#     out["brier"] = float(np.mean((pr - yt) ** 2))
#     return out


# def adjust(pvals, method):
#     """Holm (FWER) or Benjamini-Hochberg (FDR); NaNs stay NaN."""
#     p_arr = np.asarray(pvals, dtype=float)
#     out = np.full_like(p_arr, np.nan)
#     ok = np.where(np.isfinite(p_arr))[0]
#     if len(ok) == 0:
#         return out
#     order = ok[np.argsort(p_arr[ok])]
#     m = len(order)
#     if method == "holm":
#         adj = np.maximum.accumulate((m - np.arange(m)) * p_arr[order])
#     else:
#         adj = np.minimum.accumulate((m / np.arange(1, m + 1) * p_arr[order])[::-1])[::-1]
#     out[order] = np.clip(adj, 0, 1)
#     return out


# def evaluate_feature(feat):
#     """Group metrics + pairwise comparisons for ONE feature. Groups are only
#     compared with other values of the same feature (no cross-feature combinations)."""
#     rng = np.random.default_rng(CFG["seed"])        # per-feature seed -> reproducible, order-independent
#     lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
#     point, boot, meta = {}, {}, {}

#     for g, d in df.groupby(feat):
#         if len(d) < CFG["min_group_n"]:
#             continue
#         for n in names:
#             yt = d[f"y_{n}"].to_numpy().astype(int)
#             pr = d[f"pred_{n}"].to_numpy()
#             if yt.sum() < CFG["min_pos"]:
#                 continue
#             key = (g, n)
#             point[key] = metric_suite(yt, pr, CFG["threshold"])
#             meta[key] = (len(d), int(yt.sum()))
#             # resample patients WITHIN the group; disjoint groups -> independent bootstrap distributions
#             draws = {m: np.empty(CFG["n_boot"]) for m in CFG["metrics"]}
#             for b in range(CFG["n_boot"]):
#                 idx = rng.integers(0, len(d), len(d))
#                 r = metric_suite(yt[idx], pr[idx], CFG["threshold"])
#                 for m in CFG["metrics"]:
#                     draws[m][b] = r[m]
#             boot[key] = draws

#     # ---- per-group table with CIs ----
#     rows = []
#     for (g, n), pts in point.items():
#         for m in CFG["metrics"]:
#             arr = boot[(g, n)][m]; arr = arr[np.isfinite(arr)]
#             ok = len(arr) >= CFG["min_valid_frac"] * CFG["n_boot"]
#             lo, hi = np.percentile(arr, [lo_q, hi_q]) if ok else (np.nan, np.nan)
#             rows.append({"feature": feat, "group": g, "class": n, "metric": m,
#                          "n": meta[(g, n)][0], "n_pos": meta[(g, n)][1],
#                          "estimate": pts[m], "ci_lo": lo, "ci_hi": hi})
#     group_metrics = pd.DataFrame(rows)

#     # ---- pairwise comparisons; correction family = all pairs for one (class, metric) ----
#     comp = []
#     for n in names:
#         groups = sorted({k[0] for k in point if k[1] == n})
#         for m in CFG["metrics"]:
#             fam = []
#             for i in range(len(groups)):
#                 for j in range(i + 1, len(groups)):
#                     ka, kb = (groups[i], n), (groups[j], n)
#                     d_ = boot[ka][m] - boot[kb][m]
#                     d_ = d_[np.isfinite(d_)]
#                     if len(d_) < CFG["min_valid_frac"] * CFG["n_boot"]:
#                         continue
#                     lo, hi = np.percentile(d_, [lo_q, hi_q])
#                     # two-sided bootstrap p; +1 keeps p > 0 at finite n_boot
#                     p_two = min(1.0, 2 * (min((d_ <= 0).sum(), (d_ >= 0).sum()) + 1) / (len(d_) + 1))
#                     fam.append({"feature": feat, "class": n, "metric": m,
#                                 "group_a": groups[i], "group_b": groups[j],
#                                 "est_a": point[ka][m], "est_b": point[kb][m],
#                                 "diff_a_minus_b": point[ka][m] - point[kb][m],
#                                 "ci_lo": lo, "ci_hi": hi, "p_boot": p_two})
#             if fam:
#                 for r, pa in zip(fam, adjust([r["p_boot"] for r in fam], CFG["correction"])):
#                     r["p_adj"] = pa
#                 comp.extend(fam)
#     comparisons = pd.DataFrame(comp)
#     if len(comparisons):
#         comparisons["significant"] = comparisons["p_adj"] < CFG["alpha"]
#     return group_metrics, comparisons

from sklearn.metrics import roc_auc_score, average_precision_score

# Same CFG as before; only the entries this cell uses are overridden here.
CFG["metrics"] = ["auroc", "auprc", "f1", "balanced_acc", "brier"]
CFG["min_pos"] = 5   # EVERY class needs at least this many positives in a group, because
                     # the macro average needs all classes defined for AUROC/AUPRC

def macro_metrics(Y, P, thr):
    """Macro-average each metric over classes for one set of patients.
    Y, P: (n_patients, n_classes). AUROC/AUPRC are NaN if ANY class has only one
    label present (e.g. in a resample), so the macro average always covers the
    same classes. Averaging over whichever classes happen to be defined would
    change the quantity from draw to draw."""
    per = {m: [] for m in CFG["metrics"]}
    for c in range(Y.shape[1]):
        yt, pr = Y[:, c], P[:, c]
        npos = yt.sum()
        both = npos > 0 and npos < len(yt)
        per["auroc"].append(roc_auc_score(yt, pr) if both else np.nan)
        per["auprc"].append(average_precision_score(yt, pr) if both else np.nan)
        yh = pr >= thr
        tp = (yh & (yt == 1)).sum(); fp = (yh & (yt == 0)).sum()
        fn = (~yh & (yt == 1)).sum(); tn = (~yh & (yt == 0)).sum()
        sens = tp / (tp + fn) if (tp + fn) else np.nan
        spec = tn / (tn + fp) if (tn + fp) else np.nan
        per["f1"].append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan)
        per["balanced_acc"].append((sens + spec) / 2)
        per["brier"].append(np.mean((pr - yt) ** 2))
    # plain mean (not nanmean): any undefined class makes the macro value NaN
    return {m: float(np.mean(v)) for m, v in per.items()}


def adjust(pvals, method):
    """Holm (FWER) or Benjamini-Hochberg (FDR); NaNs stay NaN."""
    p_arr = np.asarray(pvals, dtype=float)
    out = np.full_like(p_arr, np.nan)
    ok = np.where(np.isfinite(p_arr))[0]
    if len(ok) == 0:
        return out
    order = ok[np.argsort(p_arr[ok])]
    m = len(order)
    if method == "holm":
        adj = np.maximum.accumulate((m - np.arange(m)) * p_arr[order])
    else:
        adj = np.minimum.accumulate((m / np.arange(1, m + 1) * p_arr[order])[::-1])[::-1]
    out[order] = np.clip(adj, 0, 1)
    return out


def evaluate_feature(feat):
    """Macro metrics per group of ONE feature, plus pairwise bootstrap comparisons
    between groups of that same feature."""
    rng = np.random.default_rng(CFG["seed"])   # fixed seed per feature -> reproducible
    lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
    ycols = [f"y_{n}" for n in names]
    pcols = [f"pred_{n}" for n in names]
    point, boot, meta, skipped = {}, {}, {}, []

    for g, d in df.groupby(feat):
        Y = d[ycols].to_numpy().astype(int)
        P = d[pcols].to_numpy()
        if len(d) < CFG["min_group_n"] or (Y.sum(axis=0) < CFG["min_pos"]).any():
            skipped.append((g, len(d)))
            continue
        point[g] = macro_metrics(Y, P, CFG["threshold"])
        meta[g] = (len(d), Y.sum(axis=0).tolist())
        # Resample PATIENTS within the group (all classes together, so each patient's
        # outcomes stay linked). Groups are disjoint -> independent bootstrap
        # distributions, which is what lets us difference them later.
        draws = {m: np.empty(CFG["n_boot"]) for m in CFG["metrics"]}
        for b in range(CFG["n_boot"]):
            idx = rng.integers(0, len(d), len(d))
            r = macro_metrics(Y[idx], P[idx], CFG["threshold"])
            for m in CFG["metrics"]:
                draws[m][b] = r[m]
        boot[g] = draws

    if skipped:
        print(f"[{feat}] skipped (too small or a class below min_pos): {skipped}")

    # ---- per-group table ----
    rows = []
    for g, pts in point.items():
        for m in CFG["metrics"]:
            arr = boot[g][m]; arr = arr[np.isfinite(arr)]
            ok = len(arr) >= CFG["min_valid_frac"] * CFG["n_boot"]
            lo, hi = np.percentile(arr, [lo_q, hi_q]) if ok else (np.nan, np.nan)
            rows.append({"feature": feat, "group": g, "metric": m, "n": meta[g][0],
                         "n_pos_per_class": meta[g][1], "estimate": pts[m],
                         "ci_lo": lo, "ci_hi": hi})
    group_metrics = pd.DataFrame(rows)

    # ---- pairwise comparisons; correction family = all group pairs for one metric ----
    groups = sorted(point)
    comp = []
    for m in CFG["metrics"]:
        fam = []
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                a, b_ = groups[i], groups[j]
                d_ = boot[a][m] - boot[b_][m]
                d_ = d_[np.isfinite(d_)]
                if len(d_) < CFG["min_valid_frac"] * CFG["n_boot"]:
                    continue
                lo, hi = np.percentile(d_, [lo_q, hi_q])
                # two-sided bootstrap p; the +1 keeps p > 0 at finite n_boot
                p_two = min(1.0, 2 * (min((d_ <= 0).sum(), (d_ >= 0).sum()) + 1) / (len(d_) + 1))
                fam.append({"feature": feat, "metric": m, "group_a": a, "group_b": b_,
                            "est_a": point[a][m], "est_b": point[b_][m],
                            "diff_a_minus_b": point[a][m] - point[b_][m],
                            "ci_lo": lo, "ci_hi": hi, "p_boot": p_two})
        if fam:
            for r, pa in zip(fam, adjust([r["p_boot"] for r in fam], CFG["correction"])):
                r["p_adj"] = pa
            comp.extend(fam)
    comparisons = pd.DataFrame(comp)
    if len(comparisons):
        comparisons["significant"] = comparisons["p_adj"] < CFG["alpha"]
    return group_metrics, comparisons

def show(feat):
    gm, cmp_ = evaluate_feature(feat)
    pd.set_option("display.width", 220)
    print(gm.round(4).to_string(index=False))
    if len(cmp_):
        print(f"\n{len(cmp_)} comparisons, {int(cmp_['significant'].sum())} significant "
              f"after {CFG['correction']} at alpha={CFG['alpha']}")
        print(cmp_[cmp_["significant"]].sort_values(["metric", "p_adj"]).round(4).to_string(index=False))
    return gm, cmp_

In [0]:
race_gm, race_cmp = show("race")

In [0]:
eth_gm, eth_cmp = show("ethnicity")

In [0]:
# "Missing" is a data-quality bucket, not a site, so drop it before comparing sites
# (swap in a filtered df if you want that; otherwise it is treated as its own group)
site_gm, site_cmp = show("care_site_id")

In [0]:
# maternal age subgroup eval
# ============================================================================
# Advanced maternal age (AMA) subgroup eval
# Requires: df, show() from the earlier cells (run BEFORE the cell that redefines CFG).
# ============================================================================
CFG.update({
    "ama_age_cutoff": 35,                               # AMA = age >= 35 at episode start
    "ama_start_cols": ["recorded_episode_start", "inferred_episode_start"],
    "ama_dob_table": CFG["cohort_table"],               # ASSUMPTION: date_of_birth lives in the cohort table;
    "ama_dob_col": "date_of_birth",                     # if not, point these at the right table/column
})

# --- pull dates from Spark for the test patients only ---
cohort_sdf = spark.table(CFG["cohort_table"])
starts = (cohort_sdf.filter(cohort_sdf.person_id.isin(pid_int))
          .select("person_id", *CFG["ama_start_cols"]).toPandas())
dob_sdf = spark.table(CFG["ama_dob_table"])
dob = (dob_sdf.filter(dob_sdf.person_id.isin(pid_int))
       .select("person_id", CFG["ama_dob_col"]).toPandas()
       .rename(columns={CFG["ama_dob_col"]: "date_of_birth"}))

# one row per person so the merge cannot duplicate patients and inflate n
starts = starts.drop_duplicates("person_id")
dob = dob.drop_duplicates("person_id")
ama = starts.merge(dob, on="person_id", how="left")

for c in CFG["ama_start_cols"] + ["date_of_birth"]:
    ama[c] = pd.to_datetime(ama[c], errors="coerce").dt.normalize()

# Reference date = earlier of the two episode starts. DataFrame.min(axis=1) skips NaT,
# so a patient with only one of the two dates still gets an age instead of dropping out
# (their age is NaN only if both starts, or the DOB, are missing).
ama["ref_start"] = ama[CFG["ama_start_cols"]].min(axis=1)

# Exact completed years: year difference, minus 1 if the birthday hasn't happened yet
# in the reference year. Cleaner than days/365.25, which can misclassify someone
# within a day or two of their 35th birthday.
before_bday = (ama["ref_start"].dt.month * 100 + ama["ref_start"].dt.day) < \
              (ama["date_of_birth"].dt.month * 100 + ama["date_of_birth"].dt.day)
ama["age_at_episode_start"] = ama["ref_start"].dt.year - ama["date_of_birth"].dt.year - before_bday.astype(int)
ama.loc[ama["ref_start"].isna() | ama["date_of_birth"].isna(), "age_at_episode_start"] = np.nan

# 0/1 flag; nullable Int64 keeps missing ages as <NA> (they are excluded from the
# groupby in show(), not silently coerced into the <35 group)
ama["advanced_maternal_age"] = (ama["age_at_episode_start"] >= CFG["ama_age_cutoff"]).astype("Int64")
ama.loc[ama["age_at_episode_start"].isna(), "advanced_maternal_age"] = pd.NA

# left-merge onto df (unique person_id on both sides) so row order still matches pids
n_before = len(df)
df = df.drop(columns=["age_at_episode_start", "advanced_maternal_age"], errors="ignore") \
       .merge(ama[["person_id", "age_at_episode_start", "advanced_maternal_age"]],
              on="person_id", how="left")
assert len(df) == n_before == len(pids)

# --- sanity checks ---
print(df["advanced_maternal_age"].value_counts(dropna=False).to_string())
print(df["age_at_episode_start"].describe().round(1).to_string())
n_missing = int(df["advanced_maternal_age"].isna().sum())
if n_missing:
    print(f"WARNING: {n_missing} patients have no computable age and are excluded from the AMA comparison")
implausible = df["age_at_episode_start"].lt(10) | df["age_at_episode_start"].gt(60)
if implausible.any():
    print(f"WARNING: {int(implausible.sum())} patients with age <10 or >60; check DOB / episode dates")

# --- subgroup eval, same pipeline as race / ethnicity / care_site_id ---
ama_gm, ama_cmp = show("advanced_maternal_age")

In [0]:
# # ============================================================================
# # Outcome-specific metrics on the FULL held-out test set (no subgroups)
# # Requires: names, p, y, CFG from cell 1; class_metrics() from the previous cell.
# # ============================================================================
# def class_metrics(yt, pr, thr):
#     """Metrics for ONE outcome class on one set of patients. NaN where undefined."""
#     out = {m: np.nan for m in CFG["metrics"]}
#     npos = yt.sum()
#     if 0 < npos < len(yt):                    # AUROC/AUPRC need both labels present
#         out["auroc"] = roc_auc_score(yt, pr)
#         out["auprc"] = average_precision_score(yt, pr)
#     yh = pr >= thr
#     tp = (yh & (yt == 1)).sum(); fp = (yh & (yt == 0)).sum()
#     fn = (~yh & (yt == 1)).sum(); tn = (~yh & (yt == 0)).sum()
#     sens = tp / (tp + fn) if (tp + fn) else np.nan
#     spec = tn / (tn + fp) if (tn + fp) else np.nan
#     out["f1"] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
#     out["balanced_acc"] = (sens + spec) / 2   # NaN if either is undefined
#     out["brier"] = float(np.mean((pr - yt) ** 2))
#     return out


# rng = np.random.default_rng(CFG["seed"])
# lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
# n_pat = len(p)

# rows = []
# for c, n in enumerate(names):
#     yt = y[:, c].astype(int)
#     pr = p[:, c]
#     est = class_metrics(yt, pr, CFG["threshold"])

#     # resample patients with replacement; recompute every metric per draw
#     draws = {m: np.empty(CFG["n_boot"]) for m in CFG["metrics"]}
#     for b in range(CFG["n_boot"]):
#         idx = rng.integers(0, n_pat, n_pat)
#         r = class_metrics(yt[idx], pr[idx], CFG["threshold"])
#         for m in CFG["metrics"]:
#             draws[m][b] = r[m]

#     for m in CFG["metrics"]:
#         arr = draws[m][np.isfinite(draws[m])]
#         ok = len(arr) >= CFG["min_valid_frac"] * CFG["n_boot"]
#         lo, hi = np.percentile(arr, [lo_q, hi_q]) if ok else (np.nan, np.nan)
#         rows.append({"class": n, "metric": m, "n": n_pat, "n_pos": int(yt.sum()),
#                      "prevalence": yt.mean(), "estimate": est[m],
#                      "ci_lo": lo, "ci_hi": hi})

# outcome_metrics = pd.DataFrame(rows)
# pd.set_option("display.width", 200)

# # wide view: one row per outcome, "estimate [lo, hi]" per metric
# wide = outcome_metrics.assign(
#     txt=lambda d: d.apply(lambda r: f"{r['estimate']:.3f} [{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]", axis=1)
# ).pivot(index="class", columns="metric", values="txt")[CFG["metrics"]]
# print(wide.to_string())
# print()
# print(outcome_metrics[["class", "n_pos", "prevalence"]].drop_duplicates().round(4).to_string(index=False))

# ============================================================================
# Outcome-specific metrics on the FULL held-out test set (no subgroups),
# plus pairwise comparisons BETWEEN outcome classes (Holm-corrected).
# Requires: names, p, y, CFG from cell 1; adjust() from the subgroup-eval cell.
# ============================================================================
def class_metrics(yt, pr, thr):
    """Metrics for ONE outcome class on one set of patients. NaN where undefined."""
    out = {m: np.nan for m in CFG["metrics"]}
    npos = yt.sum()
    if 0 < npos < len(yt):                    # AUROC/AUPRC need both labels present
        out["auroc"] = roc_auc_score(yt, pr)
        out["auprc"] = average_precision_score(yt, pr)
    yh = pr >= thr
    tp = (yh & (yt == 1)).sum(); fp = (yh & (yt == 0)).sum()
    fn = (~yh & (yt == 1)).sum(); tn = (~yh & (yt == 0)).sum()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    out["f1"] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
    out["balanced_acc"] = (sens + spec) / 2   # NaN if either is undefined
    out["brier"] = float(np.mean((pr - yt) ** 2))
    return out


rng = np.random.default_rng(CFG["seed"])
lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
n_pat = len(p)

# --- point estimates ---
est = {n: class_metrics(y[:, c].astype(int), p[:, c], CFG["threshold"])
       for c, n in enumerate(names)}

# --- PAIRED bootstrap: ONE patient resample per draw, shared by all classes. ---
# The classes are multilabel (the same patients carry several labels), so their
# metrics are correlated. Differencing draws from the same resample preserves that
# correlation; independent resamples per class would ignore it and misstate the
# variance of the differences.
draws = {n: {m: np.empty(CFG["n_boot"]) for m in CFG["metrics"]} for n in names}
for b in range(CFG["n_boot"]):
    idx = rng.integers(0, n_pat, n_pat)
    for c, n in enumerate(names):
        r = class_metrics(y[idx, c].astype(int), p[idx, c], CFG["threshold"])
        for m in CFG["metrics"]:
            draws[n][m][b] = r[m]

# --- per-class table (same output as before) ---
rows = []
for c, n in enumerate(names):
    yt = y[:, c].astype(int)
    for m in CFG["metrics"]:
        arr = draws[n][m][np.isfinite(draws[n][m])]
        ok = len(arr) >= CFG["min_valid_frac"] * CFG["n_boot"]
        lo, hi = np.percentile(arr, [lo_q, hi_q]) if ok else (np.nan, np.nan)
        rows.append({"class": n, "metric": m, "n": n_pat, "n_pos": int(yt.sum()),
                     "prevalence": yt.mean(), "estimate": est[n][m],
                     "ci_lo": lo, "ci_hi": hi})

outcome_metrics = pd.DataFrame(rows)
pd.set_option("display.width", 200)

# wide view: one row per outcome, "estimate [lo, hi]" per metric
wide = outcome_metrics.assign(
    txt=lambda d: d.apply(lambda r: f"{r['estimate']:.3f} [{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]", axis=1)
).pivot(index="class", columns="metric", values="txt")[CFG["metrics"]]
print(wide.to_string())
print()
print(outcome_metrics[["class", "n_pos", "prevalence"]].drop_duplicates().round(4).to_string(index=False))

# --- pairwise class comparisons; Holm family = all class pairs for one metric ---
# (same family definition as the subgroup evals: 3 classes -> 3 pairs per metric)
comp = []
for m in CFG["metrics"]:
    fam = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b_ = names[i], names[j]
            d_ = draws[a][m] - draws[b_][m]           # paired: same resample per draw
            d_ = d_[np.isfinite(d_)]
            if len(d_) < CFG["min_valid_frac"] * CFG["n_boot"]:
                continue
            lo, hi = np.percentile(d_, [lo_q, hi_q])
            # two-sided bootstrap p; the +1 keeps p > 0 at finite n_boot
            p_two = min(1.0, 2 * (min((d_ <= 0).sum(), (d_ >= 0).sum()) + 1) / (len(d_) + 1))
            fam.append({"metric": m, "class_a": a, "class_b": b_,
                        "est_a": est[a][m], "est_b": est[b_][m],
                        "diff_a_minus_b": est[a][m] - est[b_][m],
                        "ci_lo": lo, "ci_hi": hi, "p_boot": p_two})
    if fam:
        for r, pa in zip(fam, adjust([r["p_boot"] for r in fam], CFG["correction"])):
            r["p_adj"] = pa
        comp.extend(fam)

outcome_cmp = pd.DataFrame(comp)
if len(outcome_cmp):
    outcome_cmp["significant"] = outcome_cmp["p_adj"] < CFG["alpha"]
    print(f"\n{len(outcome_cmp)} class comparisons, {int(outcome_cmp['significant'].sum())} significant "
          f"after {CFG['correction']} at alpha={CFG['alpha']}")
    print(outcome_cmp.sort_values(["metric", "p_adj"]).round(4).to_string(index=False))

In [0]:
# ============================================================================
# Original stored test metrics (from the pickle) vs. recomputed per-outcome estimates
# Requires: exp, names, outcome_metrics from the earlier cells.
# ============================================================================
tm = exp["test_metrics"]
pd.set_option("display.width", 200)

# 1) everything stored, printed as-is (macro + per-class)
print("=== Stored test_metrics (original) ===")
for k, v in tm.items():
    if isinstance(v, (int, float, np.floating)):
        print(f"  {k:<34} {v:.4f}")

# 2) per-class stored metrics next to the recomputed estimates
#    (stored key format is f"{metric}_{class}", e.g. auroc_gdm; "balanced_acc" is stored as "bacc")
stored_key = {"auroc": "auroc", "auprc": "auprc", "f1": "f1", "balanced_acc": "bacc", "brier": "brier"}
rows = []
for n in names:
    for m in CFG["metrics"]:
        k = f"{stored_key[m]}_{n}"
        rec = outcome_metrics[(outcome_metrics["class"] == n) & (outcome_metrics["metric"] == m)]["estimate"].iloc[0]
        rows.append({"class": n, "metric": m, "stored": tm.get(k, np.nan), "recomputed": rec})
# cmp_tbl = pd.DataFrame(rows)
# cmp_tbl["abs_diff"] = (cmp_tbl["stored"] - cmp_tbl["recomputed"]).abs()

# print("\n=== Stored vs recomputed, per outcome ===")
# print(cmp_tbl.round(4).to_string(index=False))

In [0]:
%sql
-- distribution of care_site_id across training, validation, and held out test set

SELECT
    COUNT(*)                                                        AS n_patients,
    COUNT(DISTINCT p.care_site_id)                                  AS n_distinct_care_sites,  -- COUNT DISTINCT ignores NULLs
    SUM(CASE WHEN p.care_site_id IS NULL THEN 1 ELSE 0 END)         AS n_null_care_site
FROM catalog.schema.patient_clmbr_embeddings_new_v5 e
JOIN catalog.omop.person p
    ON e.patient_id = p.person_id

In [0]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

CFG = {
    "seed": 42,                              # must match the modeling split
    "test_frac": 0.10,                       # first split: carve off 10% test
    "val_frac_of_remaining": 0.10 / 0.90,    # second split: 10% of the original total
    "embeddings_table": "catalog.schema.patient_clmbr_embeddings_new_v5",
    "labels_table": "catalog.schema.person_label_mapping",
    "person_table": "catalog.omop.person",
    "unresolved_label": -1,                  # dropped in the modeling cell, so dropped here too
}

# Only patient_id is needed from the embeddings table; skipping the 768 float columns keeps this light.
# Assumption: the labels table is derived from step7_filtered_cohort_with_apos, so the inner join to
# the flags table in the modeling cell doesn't change the row set. Join order mirrors that cell
# (embeddings on the left), but row order across Spark loads is not guaranteed.
emb_ids = spark.table(CFG["embeddings_table"]).select("patient_id").toPandas()
labels_df = spark.table(CFG["labels_table"]).select("person_id", "label").toPandas()

# str keys on both sides avoid int/object merge mismatches
emb_ids["patient_id"] = emb_ids["patient_id"].astype(str)
labels_df["person_id"] = labels_df["person_id"].astype(str)

ids_df = emb_ids.merge(labels_df, left_on="patient_id", right_on="person_id", how="inner")
ids_df = ids_df[ids_df["label"] != CFG["unresolved_label"]].reset_index(drop=True)

patient_ids = ids_df["patient_id"].values
combo_label = ids_df["label"].values

# Same two-stage stratified split as the modeling cell: 80/10/10
ids_temp, ids_test, combo_temp, _ = train_test_split(
    patient_ids, combo_label, test_size=CFG["test_frac"],
    stratify=combo_label, random_state=CFG["seed"],
)
ids_train, ids_val, _, _ = train_test_split(
    ids_temp, combo_temp, test_size=CFG["val_frac_of_remaining"],
    stratify=combo_temp, random_state=CFG["seed"],
)

split_df = pd.concat([
    pd.DataFrame({"patient_id": ids_train, "split": "train"}),
    pd.DataFrame({"patient_id": ids_val,   "split": "val"}),
    pd.DataFrame({"patient_id": ids_test,  "split": "test"}),
])

# care_site_id from person
care_df = spark.table(CFG["person_table"]).select("person_id", "care_site_id").toPandas()
care_df["person_id"] = care_df["person_id"].astype(str)
split_df = split_df.merge(care_df, left_on="patient_id", right_on="person_id", how="left")

# distinct care sites per split (NaN excluded from nunique, tracked separately)
summary = split_df.groupby("split").agg(
    n_patients=("patient_id", "size"),
    n_distinct_care_sites=("care_site_id", "nunique"),
    n_null_care_site=("care_site_id", lambda s: s.isna().sum()),
).reindex(["train", "val", "test"])
print(summary)
print(f"\nDistinct care sites across all splits: {split_df['care_site_id'].nunique()}")

# care_site_id x split counts (-1 row = missing care site)
dist = pd.crosstab(split_df["care_site_id"].fillna(-1), split_df["split"])[["train", "val", "test"]]
dist["total"] = dist.sum(axis=1)
print(dist.sort_values("total", ascending=False).to_string())

# --- site overlap between splits (NaN care sites excluded: "missing" is not a site) ---
sites = {s: set(split_df.loc[split_df["split"] == s, "care_site_id"].dropna()) for s in ["train", "val", "test"]}

# sites present in val/test but never seen in train (original check)
for s in ["val", "test"]:
    print(f"{s}: {len(sites[s] - sites['train'])} care sites not seen in train")

# test sites absent from BOTH train and val, i.e. sites the model never saw in training or model selection.
# This is the closest this split gets to "unseen site"; the split is by patient, not by site.
new_in_test = sites["test"] - (sites["train"] | sites["val"])
test_rows = split_df[split_df["split"] == "test"]
n_test_pat_new = int(test_rows["care_site_id"].isin(new_in_test).sum())
print(f"\ntest: {len(new_in_test)} care sites not seen in train or val "
      f"({len(new_in_test) / len(sites['test']) * 100:.1f}% of test sites), "
      f"covering {n_test_pat_new} test patients "
      f"({n_test_pat_new / len(test_rows) * 100:.2f}% of the test set)")

In [0]:
%sql
-- distribution of visit number and record length stratified by outcome, in this cohort (taking person_id from embeddings table)
WITH cohort AS (
    -- person list comes from the embeddings table; outcome flags and episode dates from the phenotyping table
    SELECT
        e.patient_id AS person_id,
        pe.is_control, pe.has_gdm, pe.has_ghtn, pe.has_pe, pe.has_inferred_pe,
        DATEDIFF(
            GREATEST(pe.inferred_episode_end, pe.recorded_episode_end),
            LEAST(pe.inferred_episode_start, pe.recorded_episode_start)
        ) AS record_length_days
    FROM catalog.schema.patient_clmbr_embeddings_new_v5 e
    JOIN catalog.schema.step7_filtered_cohort_with_apos pe
        ON e.patient_id = pe.person_id
),
per_person AS (
    -- LEFT JOIN so patients with zero visits still count (n_visits = 0)
    SELECT
        c.person_id, c.is_control, c.has_gdm, c.has_ghtn, c.has_pe, c.has_inferred_pe,
        c.record_length_days,
        COUNT(DISTINCT vo.visit_occurrence_id) AS n_visits
    FROM cohort c
    LEFT JOIN catalog.schema.visit_occurrence_cohort vo
        ON vo.person_id = c.person_id
    GROUP BY c.person_id, c.is_control, c.has_gdm, c.has_ghtn, c.has_pe, c.has_inferred_pe, c.record_length_days
),
by_outcome AS (
    -- multilabel: a patient with both GDM and PE/GHTN appears in both rows
    SELECT 'control' AS outcome, person_id, n_visits, record_length_days FROM per_person WHERE is_control = 1
    UNION ALL
    SELECT 'gdm', person_id, n_visits, record_length_days FROM per_person WHERE has_gdm = 1
    UNION ALL
    SELECT 'pe_ghtn', person_id, n_visits, record_length_days FROM per_person
    WHERE has_ghtn = 1 OR has_pe = 1 OR has_inferred_pe = 1
)
SELECT
    outcome,
    COUNT(*)                                                         AS n_patients,
    -- visits per patient
    MIN(n_visits)                                                    AS visits_min,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY n_visits)           AS visits_p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY n_visits)           AS visits_median,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY n_visits)           AS visits_p75,
    MAX(n_visits)                                                    AS visits_max,
    ROUND(AVG(n_visits), 2)                                          AS visits_mean,
    -- record length in days
    MIN(record_length_days)                                          AS len_min,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY record_length_days) AS len_p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY record_length_days) AS len_median,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY record_length_days) AS len_p75,
    MAX(record_length_days)                                          AS len_max,
    ROUND(AVG(record_length_days), 1)                                AS len_mean
FROM by_outcome
GROUP BY outcome
ORDER BY outcome

In [0]:
# # ============================================================================
# # FULL-RECORD performance of the Optuna-best XGBoost config (self-contained: needs only Spark, the v2 tables / matrix cache, and best_params.json)
# #   STORED by mode=tune (best_params.json): the best trial's VALIDATION macro + per-label AUC, its tree counts, the config and the data sizes.
# #   RECOMPUTED here: refit the three binary models with the best config on the full-record TRAIN rows (no landmark censoring, the global selected
# #   feature set), early-stopped on validation exactly as in the study, then the full metric suite on validation (reference) and on the
# #   held-out TEST split, with bootstrap CIs on test.
# # ============================================================================
# import os, json, time, shutil
# import numpy as np
# import pandas as pd
# import scipy.sparse as sp
# import xgboost as xgb
# from joblib import Parallel, delayed
# from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, brier_score_loss, balanced_accuracy_score)

# CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
# OUTCOME_NAMES = ["pe_ghtn", "gdm"]
# LABEL_COLS = ["y_control", "y_pe_ghtn", "y_gdm"]


# def _pick_local_base():
#     for base in ("/local_disk0", "/tmp"):
#         if os.path.isdir(base) and os.access(base, os.W_OK):
#             return base
#     return "/tmp"


# CFG = {
#     "hpo_dir": "/path/to/exports/xgb_v2_hpo",                      # best_params.json written by mode=tune of the XGBoost v2 notebook
#     "tabular_meta_path": "/path/to/exports/tabular_v2/meta.json",
#     "selected_table": "catalog.schema.tabular_v2_selected_features",
#     "split_table": "catalog.schema.protopregnet_landmark_cohort",
#     "cache_dir": "/path/to/exports/tabular_v2_cache",
#     "local_cache_dir": f"{_pick_local_base()}/tabular_v2_cache",
#     "n_jobs": 32,                                                                      # XGBoost threads and the bootstrap
#     "n_bootstrap": 1000, "bootstrap_chunks": 8, "ci": 0.95, "bootstrap_seed": 42,     # same as the landmark runs
#     # NOT stored in best_params.json: must equal CFG['xgb_fixed'] of the XGBoost v2 notebook (copied from it). The check in section 3 flags a mismatch.
#     "xgb_fixed": {
#         "objective": "binary:logistic", "tree_method": "hist",   # CPU histogram algorithm
#         "eval_metric": "auc",         # early stopping watches validation AUC (the metric we report); the LAST eval metric is the one used
#         "max_bin": 256,
#         "n_estimators": 1000,         # a cap: early stopping picks the real number of trees per label and per landmark
#         "early_stopping_rounds": 50, "random_state": 42, "verbosity": 0,
#     },
# }

# # ===== helpers: copied unchanged from the XGBoost v2 notebook =====
# CACHE_FILES = ["X_data.npy", "X_indices.npy", "X_indptr.npy", "person_index.npy", "feature_names.json", "meta.json"]


# def _cp_from_volume(volume_path, local_path):
#     try:
#         dbutils.fs.cp(volume_path, f"file:{local_path}")
#     except Exception:
#         shutil.copyfile(volume_path, local_path)


# def read_meta(directory):
#     with open(f"{directory}/meta.json") as f:
#         return json.load(f)


# def ensure_local_cache():
#     """The matrix lives on the node's local disk so concurrent tasks can memory-map ONE copy (shared page cache). If the node has no
#     local copy (e.g. the cluster restarted), copy it from the Volume once; a task that loses the race just uses the winner's copy."""
#     loc = CFG["local_cache_dir"]
#     if os.path.exists(f"{loc}/meta.json"):
#         return loc
#     if not os.path.exists(f"{CFG['cache_dir']}/meta.json"):
#         raise FileNotFoundError(f"no matrix cache in {CFG['cache_dir']} -- run the notebook once with mode=prep first")
#     tmp = f"{loc}.tmp_{os.getpid()}"
#     shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
#     for fn in CACHE_FILES:
#         _cp_from_volume(f"{CFG['cache_dir']}/{fn}", f"{tmp}/{fn}")
#     try:
#         os.rename(tmp, loc)
#     except OSError:
#         shutil.rmtree(tmp, ignore_errors=True)                        # another task finished first
#     return loc


# def load_cache():
#     loc = ensure_local_cache()
#     with open(f"{loc}/meta.json") as f:
#         meta = json.load(f)
#     data, indices = np.load(f"{loc}/X_data.npy", mmap_mode="r"), np.load(f"{loc}/X_indices.npy", mmap_mode="r")
#     X = sp.csr_matrix((data, indices, np.load(f"{loc}/X_indptr.npy")), shape=tuple(meta["shape"]), copy=False)   # read-only, memory-mapped
#     with open(f"{loc}/feature_names.json") as f:
#         feature_names = json.load(f)
#     return X, feature_names, np.load(f"{loc}/person_index.npy")


# def matrix_fingerprint(feature_names, shape):
#     """Identifies one built matrix, so the selected-features table and every cache can be checked to belong together."""
#     import hashlib
#     h = hashlib.sha1(("|".join(feature_names) + f"#{shape[0]}x{shape[1]}").encode()).hexdigest()
#     return h[:16]


# def build_design_spec(sel, feature_names):
#     """sel: the selected rows of tabular_v2_selected_features. Restricts the stored matrix to the base columns that are needed and maps every design
#     column to its (restricted) base column."""
#     col_index = {n: i for i, n in enumerate(feature_names)}
#     missing = [c for c in sel["base_col"] if c not in col_index]
#     assert not missing, f"{len(missing)} selected base columns are not in the cached matrix, e.g. {missing[:3]}: rebuild the cache (mode=prep)"
#     base_full = sel["base_col"].map(col_index).to_numpy()
#     uniq = np.unique(base_full)
#     return {"uniq": uniq, "base_idx": np.searchsorted(uniq, base_full), "is_presence": (sel["role"] == "presence").to_numpy(),
#             "impute": sel["impute_value"].to_numpy(dtype=float), "design_cols": sel["design_col"].tolist()}


# def make_design_dense(Xc, base_idx, is_presence, impute):
#     """Dense float32 design for the SELECTED design columns, in the order given.
#       presence column: 1 if the stored base column has an entry (structure), else 0
#       value column:    the stored value if present, else impute (train mean of the present values)
#     Mean-imputation + the presence column is the standard 'value + indicator' encoding; trees are invariant to the constant, so no scaling or
#     centering is applied here."""
#     n = Xc.shape[0]
#     sub = Xc[:, np.asarray(base_idx)].tocoo()
#     out = np.tile(np.where(is_presence, 0.0, impute).astype(np.float32), (n, 1))
#     out[sub.row, sub.col] = np.where(is_presence[sub.col], 1.0, sub.data).astype(np.float32)
#     return out


# def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
#     results = {}
#     y_pred = (y_prob >= 0.5).astype(int)
#     for c, name in enumerate(class_names):
#         yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
#         n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
#         results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
#         results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
#         results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
#         results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
#         results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

#     def macro(prefix):
#         vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
#         return float(np.mean(vals)) if vals else float("nan")

#     for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
#         results[f"{m}_macro"] = macro(m)
#     for m in ["auroc", "auprc"]:
#         vals = [results[f"{m}_{n}"] for n in outcome_names if not np.isnan(results[f"{m}_{n}"])]
#         results[f"{m}_macro_outcomes_only"] = float(np.mean(vals)) if vals else float("nan")
#     results["_primary"] = results["auroc_macro"]
#     return results


# def _boot_chunk(Y_true, Y_prob, class_names, n_iter, seed, keys):
#     rng = np.random.RandomState(seed)
#     n = Y_true.shape[0]
#     out = {k: [] for k in keys}
#     for _ in range(n_iter):
#         idx = rng.randint(0, n, n)
#         yt_r, yp_r = Y_true[idx], Y_prob[idx]
#         if any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(class_names))):
#             continue
#         m = _compute_metric_suite(yt_r, yp_r, class_names=class_names, outcome_names=OUTCOME_NAMES)
#         for k in keys:
#             out[k].append(m[k])
#     return out


# def bootstrap_metrics_ci(Y_true, Y_prob, class_names):
#     """Same procedure as the ProtoPregNet landmark runs (resample test patients, recompute the suite, percentile CI), split into a FIXED
#     number of chunks with fixed seeds so the result does not depend on n_jobs; chunks run in parallel."""
#     stems = ["auroc", "auprc", "f1", "bacc", "brier"]
#     keys = ([f"{m}_macro" for m in stems] + ["auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]
#             + [f"{m}_{c}" for m in stems for c in class_names])
#     sizes = np.full(CFG["bootstrap_chunks"], CFG["n_bootstrap"] // CFG["bootstrap_chunks"])
#     sizes[: CFG["n_bootstrap"] % CFG["bootstrap_chunks"]] += 1
#     parts = Parallel(n_jobs=min(CFG["n_jobs"], CFG["bootstrap_chunks"]))(
#         delayed(_boot_chunk)(Y_true, Y_prob, class_names, int(s), CFG["bootstrap_seed"] + i, keys) for i, s in enumerate(sizes))
#     lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
#     res = {}
#     for k in keys:
#         arr = np.concatenate([np.asarray(p[k], dtype=float) for p in parts]); arr = arr[~np.isnan(arr)]
#         res[k] = (float("nan"),) * 3 if len(arr) == 0 else (arr.mean(), *np.percentile(arr, [lo_q, hi_q]))
#     return res


# def fit_label_model(Xtr, ytr, Xva, yva, hp, n_threads):
#     """One binary XGBoost model for one label, early-stopped on the validation split.
#     hp = the tuned hyperparameters (including spw_power). scale_pos_weight is computed PER LABEL from the training prevalence, because the three
#     labels have very different class balance; spw_power (tuned) interpolates between no weighting (0) and full balancing (1)."""
#     pos = int(ytr.sum()); neg = len(ytr) - pos
#     if min(pos, neg) == 0 or int(yva.sum()) in (0, len(yva)):
#         raise ValueError(f"a label has a single class in train or validation (train pos {pos}/{len(ytr)}, val pos {int(yva.sum())}/{len(yva)})")
#     params = {**CFG["xgb_fixed"], **{k: v for k, v in hp.items() if k != "spw_power"},
#               "scale_pos_weight": (neg / pos) ** hp["spw_power"], "n_jobs": int(n_threads)}
#     model = xgb.XGBClassifier(**params)
#     model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
#     return model


# def predict_all(models, Xd):
#     """(n, 3) probabilities, columns in CLASS_NAMES order. The best iteration is passed explicitly so the result does not depend on the
#     xgboost version's default for early-stopped models."""
#     return np.column_stack([m.predict_proba(Xd, iteration_range=(0, m.best_iteration + 1))[:, 1] for m in models])


# # ===== 1. what is already stored =====
# path = f"{CFG['hpo_dir']}/best_params.json"
# if not os.path.exists(path):
#     raise FileNotFoundError(f"{path} not found: run mode=tune of the XGBoost v2 notebook first")
# with open(path) as f:
#     BP = json.load(f)
# CFG["xgb_fixed"] = BP.get("xgb_fixed", CFG["xgb_fixed"])                              # future-proof: used if a later notebook version stores it
# print(f"STORED in best_params.json: study '{BP['study_name']}', {BP['n_trials_complete']} complete / {BP['n_trials_pruned']} pruned trials (final={BP['final']})")
# print(f"  validation macro AUC of the best trial {BP['best_value']:.4f} | by label: {({k: round(v, 4) for k, v in BP['val_auc_by_label'].items()})}")
# print(f"  early-stopped tree counts of the best trial: {({k: v + 1 for k, v in BP['best_iteration_by_label'].items()})}")
# print(f"  trained on {BP['n_train']:,} train patients, validated on {BP['n_val']:,}, {BP['n_design_columns']:,} design columns, xgboost {BP.get('xgboost')} (now {xgb.__version__})")
# print(f"  best params: {BP['best_params']}")
# print("  NOT stored: test metrics, AUPRC / F1 / balanced accuracy / Brier, confidence intervals, the fitted models, the fixed xgb settings -> recomputed / checked below\n")

# # ===== 2. data: the v2 matrix, the GLOBAL selected feature set, the ProtoPregNet split (same code path as the XGBoost v2 notebook) =====
# t0 = time.time()
# X_full, feature_names, person_index = load_cache()
# pid_int_all = person_index.astype("int64")
# with open(CFG["tabular_meta_path"]) as f:
#     tmeta = json.load(f)
# FP = matrix_fingerprint(feature_names, X_full.shape)
# assert FP == tmeta["fingerprint"] == read_meta(CFG["local_cache_dir"]).get("fingerprint") == BP.get("matrix_fingerprint", FP), \
#     "cache / tabular_v2 meta.json / best_params.json disagree: the config was tuned on a different matrix"
# sel = (spark.table(CFG["selected_table"]).filter("selected_final = true")
#        .select("design_col", "role", "base_col", "impute_value", "matrix_fingerprint").toPandas())
# assert set(sel["matrix_fingerprint"]) == {FP}, "the selected-features table was built from a different matrix"
# sel = sel.sort_values(["role", "design_col"]).reset_index(drop=True)             # same column order as the notebook
# spec = build_design_spec(sel, feature_names)
# X = X_full[:, spec["uniq"]].tocsr()
# coh = spark.table(CFG["split_table"]).select("pid_int", "split", *LABEL_COLS).toPandas().set_index("pid_int")
# meta = coh.reindex(pid_int_all)
# assert meta["split"].notna().all(), "matrix patients missing from the split table"
# Y_all = meta[LABEL_COLS].to_numpy(dtype=int); split_all = meta["split"].to_numpy()
# rows = {s: np.flatnonzero(split_all == s) for s in ("train", "val", "test")}
# design = lambda r: make_design_dense(X[r], spec["base_idx"], spec["is_presence"], spec["impute"])      # FULL record: no censoring
# Xtr, Xva, Xte = design(rows["train"]), design(rows["val"]), design(rows["test"])
# Ytr, Yva, Yte = Y_all[rows["train"]], Y_all[rows["val"]], Y_all[rows["test"]]
# print(f"full-record design: train {Xtr.shape}, val {Xva.shape}, test {Xte.shape} | prevalence test (control, pe_ghtn, gdm) {Yte.mean(0).round(3).tolist()} | {time.time() - t0:.0f}s")

# # ===== 3. refit the best config (3 binary models, early-stopped on validation, as in the study) and evaluate =====
# t0 = time.time()
# models = [fit_label_model(Xtr, Ytr[:, i], Xva, Yva[:, i], BP["best_params"], CFG["n_jobs"]) for i in range(len(CLASS_NAMES))]
# re_auc = {c: float(m.best_score) for c, m in zip(CLASS_NAMES, models)}; re_iter = {c: int(m.best_iteration) for c, m in zip(CLASS_NAMES, models)}
# max_diff = max(abs(re_auc[c] - BP["val_auc_by_label"][c]) for c in CLASS_NAMES); same_iter = all(re_iter[c] == BP["best_iteration_by_label"][c] for c in CLASS_NAMES)
# print(f"refit in {time.time() - t0:.0f}s | trees per label {({c: v + 1 for c, v in re_iter.items()})} | recomputed validation AUC by label {({c: round(v, 4) for c, v in re_auc.items()})}, "
#       f"macro {np.mean(list(re_auc.values())):.4f} vs stored {BP['best_value']:.4f}")
# print(f"  max |diff| to the stored per-label validation AUC {max_diff:.1e}, same tree counts: {same_iter} -> "
#       + ("reproduces the tuned models" if max_diff < 1e-4 and same_iter else "DIFFERS - the fixed xgb settings above, the xgboost version, the matrix or the selected set may differ from the study's"))
# Pva, Pte = predict_all(models, Xva), predict_all(models, Xte)
# m_val, m_test = _compute_metric_suite(Yva, Pva), _compute_metric_suite(Yte, Pte)
# t0 = time.time(); ci = bootstrap_metrics_ci(Yte, Pte, CLASS_NAMES); print(f"bootstrap CIs ({CFG['n_bootstrap']} resamples) in {time.time() - t0:.0f}s")

# # ===== 4. result table: test (with 95% CI) and validation, same metric suite as the landmark runs =====
# out = []
# def add(scope, cls, stem, key):
#     _, lo, hi = ci[key]
#     out.append({"scope": scope, "class": cls, "metric": stem, "test": m_test[key], "test_ci_lo": lo, "test_ci_hi": hi, "val": m_val[key]})
# for stem in ["auroc", "auprc", "f1", "bacc", "brier"]:
#     add("macro", "ALL", stem, f"{stem}_macro")
#     for c in CLASS_NAMES:
#         add("class", c, stem, f"{stem}_{c}")
# for stem in ["auroc", "auprc"]:
#     add("macro_outcomes_only", "ALL", stem, f"{stem}_macro_outcomes_only")
# full_record_metrics = pd.DataFrame(out)
# pd.set_option("display.width", 200)
# print(f"\nFULL-RECORD performance of the best XGBoost config (fit on {len(rows['train']):,} train patients; test n = {len(rows['test']):,}; F1 / balanced accuracy / Brier at threshold 0.5)")
# print("the validation column is optimistic: validation chose both the tree counts (early stopping) and the hyperparameters")
# print(full_record_metrics.round(4).to_string(index=False))
# full_record_metrics

In [0]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

def holm_adjust(p):
    """Holm step-down adjusted p-values (same as multipletests(method='holm')[1])."""
    p = np.asarray(p, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        # multiply the k-th smallest p by (m - k), then force the sequence to be non-decreasing
        running_max = max(running_max, (m - rank) * p[idx])
        adj[idx] = min(1.0, running_max)
    return adj

def ols_coef_p(X, y, coef_idx):
    """Classical OLS: coefficient and two-sided t-test p-value for one column of X (X includes the intercept)."""
    n, k = X.shape
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    sigma2 = resid @ resid / (n - k)               # residual variance
    cov = sigma2 * np.linalg.inv(X.T @ X)          # classical covariance of the coefficients
    t = beta[coef_idx] / np.sqrt(cov[coef_idx, coef_idx])
    return beta[coef_idx], 2 * stats.t.sf(abs(t), df=n - k)

CFG = {
    "embeddings_table": "catalog.schema.patient_clmbr_embeddings_new_v5",
    "cohort_table": "catalog.schema.step7_filtered_cohort_with_apos",
    "visit_table": "catalog.schema.visit_occurrence_cohort",
    "metrics": ["n_visits", "record_length_days"],
    "alpha": 0.05,
    "correction": "holm",  # family-wise control across the pairwise tests
}

# Same per-person table as the SQL above, returned unaggregated so we can test on it
per_person_df = spark.sql(f"""
    WITH cohort AS (
        SELECT e.patient_id AS person_id,
               pe.is_control, pe.has_gdm, pe.has_ghtn, pe.has_pe, pe.has_inferred_pe,
               DATEDIFF(GREATEST(pe.inferred_episode_end, pe.recorded_episode_end),
                        LEAST(pe.inferred_episode_start, pe.recorded_episode_start)) AS record_length_days
        FROM {CFG['embeddings_table']} e
        JOIN {CFG['cohort_table']} pe ON e.patient_id = pe.person_id
    )
    SELECT c.*, COUNT(DISTINCT vo.visit_occurrence_id) AS n_visits
    FROM cohort c
    LEFT JOIN {CFG['visit_table']} vo ON vo.person_id = c.person_id
    GROUP BY c.person_id, c.is_control, c.has_gdm, c.has_ghtn, c.has_pe, c.has_inferred_pe, c.record_length_days
""").toPandas()
per_person_df["person_id"] = per_person_df["person_id"].astype(str)

d = per_person_df
control = d["is_control"] == 1
gdm = d["has_gdm"] == 1
pe_ghtn = (d["has_ghtn"] == 1) | (d["has_pe"] == 1) | (d["has_inferred_pe"] == 1)
print("control rows that also carry a case flag (expect 0):", int((control & (gdm | pe_ghtn)).sum()))

# Mutually exclusive groups for the omnibus test; np.select takes the first match, so control wins any conflict
d["group_exclusive"] = np.select(
    [control, gdm & ~pe_ghtn, pe_ghtn & ~gdm, gdm & pe_ghtn],
    ["control", "gdm_only", "pe_ghtn_only", "both"], default="other")
d = d[d["group_exclusive"] != "other"]  # drops patients who are neither control nor case

# Descriptives: median [IQR] per group
for m in CFG["metrics"]:
    q = d.groupby("group_exclusive")[m].quantile([0.25, 0.5, 0.75]).unstack()
    q["n"] = d.groupby("group_exclusive")[m].size()
    print(f"\n{m}\n", q.rename(columns={0.25: "p25", 0.5: "median", 0.75: "p75"}))

# Omnibus (non-parametric: counts and lengths are skewed, and visit counts have many ties)
for m in CFG["metrics"]:
    samples = [g[m].dropna().values for _, g in d.groupby("group_exclusive")]
    H, p = stats.kruskal(*samples)
    print(f"Kruskal-Wallis, {m} across exclusive groups: H={H:.1f}, p={p:.3g}")

# Pairwise vs control. Case groups use the overlapping (multilabel) definitions, as in Table 1;
# controls are disjoint from both, so each comparison is between independent samples.
rows = []
for name, mask in [("gdm", gdm), ("pe_ghtn", pe_ghtn)]:
    for m in CFG["metrics"]:
        x = per_person_df.loc[mask, m].dropna().values
        y = per_person_df.loc[control, m].dropna().values
        U, p = stats.mannwhitneyu(x, y, alternative="two-sided", method="asymptotic")
        auc = U / (len(x) * len(y))  # P(case value > control value) + 0.5*P(tie); 0.5 = no difference
        rows.append({"group": name, "metric": m, "n_case": len(x), "n_control": len(y),
                     "median_case": np.median(x), "median_control": np.median(y),
                     "auc": auc, "rank_biserial": 2 * auc - 1, "p": p})
res = pd.DataFrame(rows)
res["p_adj"] = multipletests(res["p"], alpha=CFG["alpha"], method=CFG["correction"])[1]
print(res.to_string(index=False))
# With ~60k patients nearly any shift is "significant"; read the AUC / rank-biserial column for size.

In [0]:
import pickle
import numpy as np
import pandas as pd
from scipy import stats

CFG = {
    "export_path": "/path/to/exports/full_analysis_export.pkl",  # same pickle the subgroup cells use
    "classes": ["gdm", "pe_ghtn"],       # classes to test; looked up BY NAME in exp["CLASS_NAMES"], never by position
    "metrics": ["n_visits", "record_length_days"],
    "alpha": 0.05,
}

# reuse `exp` if cell 1 is still in memory, otherwise reload it
if "exp" not in globals():
    with open(CFG["export_path"], "rb") as f:
        exp = pickle.load(f)

names = list(exp["CLASS_NAMES"])
pids = np.asarray(exp["pids_test"])
p_test = np.asarray(exp["p_test"])
y_test = np.asarray(exp["Y_fusion_test"])
assert p_test.shape == y_test.shape == (len(pids), len(names)), (p_test.shape, y_test.shape, len(pids), len(names))
assert len(set(pids.tolist())) == len(pids), "pids_test has duplicates, so it is not patient-level"

# Test-set predictions AND labels come from the export, so the labels are exactly what the model was scored against.
# str(int(...)) matches how per_person_df.person_id was cast, and drops any padding or float formatting on the pid.
pred_df = pd.DataFrame({"patient_id": [str(int(x)) for x in pids]})
for cls in CFG["classes"]:
    j = names.index(cls)                       # raises ValueError if the class name is not in the export
    pred_df[f"p_{cls}"] = p_test[:, j]
    pred_df[f"y_{cls}"] = y_test[:, j].astype(int)

# covariates (n_visits, record_length_days) come from per_person_df (built in the visit/record-length cell).
# LEFT merge so test patients missing from it are reported instead of silently dropped.
t = pred_df.merge(per_person_df, left_on="patient_id", right_on="person_id", how="left")
matched = t["person_id"].notna()
print(f"{int(matched.sum())} of {len(pred_df)} test patients matched to visit/record-length data")

# Consistency check: export labels vs labels re-derived from the phenotyping flags.
# A low agreement rate means the two use different outcome definitions.
flag_label = {
    "gdm": t["has_gdm"] == 1,
    "pe_ghtn": (t["has_ghtn"] == 1) | (t["has_pe"] == 1) | (t["has_inferred_pe"] == 1),
}
for cls in CFG["classes"]:
    if cls in flag_label:
        agree = (t.loc[matched, f"y_{cls}"] == flag_label[cls][matched].astype(int)).mean()
        print(f"{cls}: export label agrees with phenotyping flags for {agree:.4f} of matched patients")

rows = []
for cls in CFG["classes"]:
    pcol, ycol = f"p_{cls}", f"y_{cls}"
    for m in CFG["metrics"]:
        ok = t[pcol].notna() & t[m].notna()      # unmatched patients have NaN covariates and fall out here
        pred, cov, lab = t.loc[ok, pcol], t.loc[ok, m], t.loc[ok, ycol]

        # 1) raw Spearman: includes any association that runs through the outcome itself
        rho, p = stats.spearmanr(pred, cov)
        rows.append({"class": cls, "metric": m, "test": "raw", "stat": rho, "p": p, "n": int(ok.sum())})

        # 2) within-label Spearman: holds the true outcome fixed
        for val, tag in [(1, "within_label_pos"), (0, "within_label_neg")]:
            s = lab == val
            rho, p = stats.spearmanr(pred[s], cov[s])
            rows.append({"class": cls, "metric": m, "test": tag, "stat": rho, "p": p, "n": int(s.sum())})

        # 3) label-adjusted: OLS on ranks, rank(pred) ~ label + rank(covariate). Approximates a partial Spearman
        #    correlation and assumes the label shifts prediction ranks by one constant amount in both strata.
        #    columns: intercept, true label, rank of covariate; we test the covariate-rank coefficient
        X = np.column_stack([np.ones(int(ok.sum())), lab.values, stats.rankdata(cov)])
        coef, p = ols_coef_p(X, stats.rankdata(pred), coef_idx=2)
        rows.append({"class": cls, "metric": m, "test": "label_adjusted", "stat": coef, "p": p, "n": int(ok.sum())})

out = pd.DataFrame(rows)
# Holm within each family of tests, so the within-label and adjusted results are not diluted by the raw ones
out["p_adj"] = out.groupby("test")["p"].transform(
    lambda s: pd.Series(holm_adjust(s.values), index=s.index))
out["sig"] = out["p_adj"] < CFG["alpha"]
print(out.to_string(index=False))

In [0]:
# # !pip install "numpy==1.26.4"
# dbutils.library.restartPython()
